In [ ]:
from AI4AO import PyramidWFS, PhaseDataset, DeformableMirror
from AI4AO import imshow, imshow_multiple
from mmengine import Config
from kalman.Kalman_tools import *
import torch
device = "cuda"

import pylab as plt
import numpy as np

import dao

from AI4AO.paths import DATA_DIR

device = 'cuda'
PATH = DATA_DIR / "Data"

In [ ]:
M2C = np.load(PATH / r"M2C.npy").astype(np.float32)

MODAL_VECTOR_INFERENCE = "/tmp/pyrModesNN.im.shm"
CONTROL_SIGNAL = "/tmp/ctrlModes07.im.shm"
SHM_LOOP_CMD_NN = "/tmp/lpCmdNN.im.shm" # Ajouté
SHM_LOOP_GAIN = "/tmp/lpGain.im.shm"
SHM_LOOP_LEAK = "/tmp/lpLeak.im.shm"

MODAL_VECTOR_INFERENCE_SHM = dao.shm(MODAL_VECTOR_INFERENCE)
CONTROL_SIGNAL_SHM = dao.shm(CONTROL_SIGNAL)
SHM_LOOP_CMD_NN_SHM = dao.shm(SHM_LOOP_CMD_NN)
SHM_LOOP_GAIN_SHM = dao.shm(SHM_LOOP_GAIN)
SHM_LOOP_LEAK_SHM = dao.shm(SHM_LOOP_LEAK)

shm = dao.shm('/tmp/lpCmdNNModalEnable.im.shm')
shm.set_data(np.full_like(np.asarray(shm.get_data()), 0))

nModes = 70

In [ ]:
paramfile = r'/home/rama/rama-dev/foyarzun/CNN/Scripts/Rama_params.py'  # file of experimental parameters


# Config extraction
AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']
DMParams = Config.fromfile(paramfile)['DMParams']

In [ ]:
WFSParams["centralObstruction"] = 0.
wfs = PyramidWFS(WFSParams, device)
PATH_WFS = PATH / "RamaWFS.pth"
wfs.LoadCalibration(PATH_WFS)
wfs.eval()


dm = DeformableMirror(WFSParams, DMParams, device)
PATH_DM = PATH / "RamaDM.pth"
dm.LoadCalibration(PATH_DM)

modes = dm(torch.from_numpy(M2C.T).cuda())[:nModes]
wfs.modulation = 0
wfs.BuildReconstructionMatrix(modes)

In [ ]:
z_estimated = MODAL_VECTOR_INFERENCE_SHM.get_data() * 0
Nmodes = MODAL_VECTOR_INFERENCE_SHM.get_data().shape[0]

gain = SHM_LOOP_GAIN_SHM.get_data()[0,0]
leak = SHM_LOOP_LEAK_SHM.get_data()[0,0]

wfs_measurements, dm_commands = [], []

n_steps = 10000
i = 0

while 1:

    if SHM_LOOP_CMD_NN_SHM.get_data()[0,0]:
        z_output = MODAL_VECTOR_INFERENCE_SHM.get_data(check=True, semNb=3)

        z_estimated = z_estimated * leak + gain * z_output

        CONTROL_SIGNAL_SHM.set_data(-z_estimated)
        i += 1

    else:
        i = 0
        z_output = MODAL_VECTOR_INFERENCE_SHM.get_data(check=True, semNb=3)
        z_estimated *= 0
        CONTROL_SIGNAL_SHM.set_data(z_estimated)


    if i > n_steps * 0.1:
        wfs_measurements.append(z_output[:nModes])
        dm_commands.append(z_estimated[:nModes])

    if i > n_steps: break
    
kalmanData = KalmanData(
    wfs_measurements=torch.from_numpy(np.stack(wfs_measurements)).cuda().squeeze(),
    modal_dm_commands=torch.from_numpy(np.stack(dm_commands)).cuda().squeeze()
)

In [ ]:
x = estimate_atmospheric_state(kalmanData.wfs_measurements, kalmanData.modal_dm_commands, 2)
A = estimate_A(x)
R = estimate_R(wfs.reconstructionMatrix, wfs.reference_intensity, 250*34*34*3.14, 10)
Q_raw = estimate_raw_Q(x, A)
Q = estimate_Q(Q_raw, A, R)
H = torch.eye(nModes, device=device)
P0 = A @ torch.cov(x.T) @ A.T + Q

In [ ]:
matrices = KalmanMatrices(
    A=A,
    H=H,
    Q=Q,
    R=R,
)

state = KalmanState(
    x=x[-1],
    P=P0,
)

In [ ]:
z_estimated_kalman = np.copy(z_estimated)[:nModes]
z_estimated_kalman[nModes:] = 0

for i in range(3000):

    z_output = MODAL_VECTOR_INFERENCE_SHM.get_data(check=True, semNb=3)[:nModes]

    # Convert residual measurement into an atmospheric measurement
    x_measurement = torch.from_numpy(z_estimated_kalman + z_output).cuda()

    # Estimate atmosphere
    with torch.no_grad():
    # Estimate atmosphere
        state, K = kalman_filter(
            measurement=x_measurement.squeeze(),
            state=state,
            matrices=matrices,
        )

        z_estimated_kalman = (A @ (state.x)).view(-1,1).cpu().numpy()

    if np.abs(z_estimated_kalman).max() > 100:
        CONTROL_SIGNAL_SHM.set_data(z_estimated_kalman*0)
        print(i, "out")
        break
    
    CONTROL_SIGNAL_SHM.set_data(-z_estimated_kalman)
CONTROL_SIGNAL_SHM.set_data(z_estimated_kalman*0)